In [ ]:
import pandas as pd
import re

df = pd.read_csv("/content/subjects-questions.csv")

In [ ]:
df.rename(columns={'eng':'questions'},inplace=True)
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122519 entries, 0 to 122518
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   questions  122519 non-null  object
 1   Subject    122519 non-null  object
dtypes: object(2)
memory usage: 1.9+ MB


In [ ]:
pd.set_option('display.max_colwidth', None)  # Show full column width
df.sample(10)
# df.loc[52628,"questions"]


,questions,Subject
119160,The volume of an auditorium is \( 100 \mathrm{m} \times \)\n\( 50 m \times 15 m . \) The reverberation time in\nthat auditorium is 2 s when the person\nwho is giving a speech is at a distance of \( 5 \mathrm{m} \) from a wall. The reverberation\ntime if the same person gives speech at a distance 10m from the wall is\nA . \( 1 \mathrm{s} \)\nB. 2\n\( c \cdot 0.5 s \)\nD. 4 s,Physics
19402,"The domain of the function \( f(x)= \)\n\( \frac{\sqrt{5-x}}{(x+1)(x-2)(x-3)}+ \)\n\( a \sin ^{-1}\left[\frac{2 x-5}{3}\right] ; a \neq 0 \) and \( [\cdot] \) denotes\ngreatest integer function, is:\nA \( \cdot[1,5]-\{2,3\} \)\n3\nB . [1,2)\( \cup(3,5] \)\nc. \( \left[1, \frac{11}{2}\right) \)\nD・ [1,5]",Maths
25747,"Two particles are placed at some distance and the magnitude of gravitational force between them is F. If the mass of each of the two particles is doubled, keeping the distance between them unchanged, the new value of gravitational force, in terms of \( F \) between them will be:\nA \( \cdot \frac{F}{4} \)\nв. \( 4 F \)\nc. \( \frac{F}{2} \)\nD.",Physics
86416,"Which of the following is/ are correct\nconcerning ""The working principle of\nLED""?\nThis question has multiple correct options\nA. electrons travel from the n-type to the p-type and holes in the opposite direction\nB. n-type silicon has extra electrons to combine with the extra holes of p-type silicon\nC. p-type silicon has extra electrons to combine with the extra holes of n-type silicon\nD. Both A and C",Physics
87985,"Which pair of substances will have the most similar geometry?\n\( A \cdot S O_{3}, S O_{3}^{2-} \)\nв. \( S O_{3}, S O_{4}^{2-} \)\n\( \mathrm{c} \cdot \mathrm{SO}_{3}, \mathrm{CO}_{3}^{2} \)\nD. \( S O_{4}^{2-}, C O_{3}^{2} \)",Chemistry
39351,The characteristic feature of ovule of\ngymnosperms is\nA. Orthotropus\nB. Unitegmic\nc. Amphitropous\nD. Both A and B,Biology
8687,Eutrophication results in the reduction\nof\nA. Dissolved hydrogen\nB. Dissolved oxygen\nc. Mineral salts\nD. None of the above,Biology
106977,"If the atomic weight of oxygen were taken as \( 90, \) then what would be\nmolecular weight of water?\nA . 101.25\nв. 104.52\nc. 112.5\nD. 142.5",Chemistry
74051,Which of the following pair of elements do not resemble according to Newlands\narrangement?\nA. Lithium and sodium\nB. Beryllium and magnesium\nc. Boron and silicon\nD. Oxygen and selenium,Chemistry
70660,Which of the molecule does not exist?\nA \( . C l F_{3} \)\nв. \( I C l_{3} \)\n\( \mathbf{c} \cdot B r F_{3} \)\nD. \( C l I_{3} \),Chemistry


Text Preprocessing

*   visual dependency
*   Latex presentation syntax
*   Whitespace & formatting cleanup
*   List item





In [ ]:
#latex syntax clean

#Unwrapping Standard Commands
#Commands like \text{}, \mathbf{}, \boldsymbol{}, \mathrm{}, and \operatorname{} are just containers.
#We want to "unwrap" them to keep only what is inside the curly braces.



#test = df.loc[25:50, 'questions'].copy()
print(df.loc[62542,'questions'])


def normalize_latex(series, cmd):

    pattern = rf'\\{cmd}\s*\{{\s*([^}}]+?)\s*\}}'
    return series.str.replace(pattern, r'\1', regex=True)



def normalize_latex_operators(series):
    # \vec{a}  vector a
    series = series.str.replace(
        r'\\vec\s*\{([^}]+)\}',
        r'vector \1',
        regex=True
    )

    #\frac{a}{b} (a)/(b)
    series = series.str.replace(
        r'\\frac\s*\{\s*([^{}]+)\s*\}\s*\{\s*([^{}]+)\s*\}',
        r'(\1)/(\2)',
        regex=True
    )

    #cdot
    series = series.str.replace(
    r'([A-Za-z0-9\}])\s*\\cdot\s*([A-Za-z0-9\{])',
    r'\1 x \2',
    regex=True
    )

    series = series.str.replace(
    r'\b([A-Da-d])\b\s*\\cdot',
    r'\1 ',
    regex=True
    )

    # remove \left and \right completely
    series = series.str.replace(r'\\left\b', ' ', regex=True)
    series = series.str.replace(r'\\right\b', ' ', regex=True)


    #times
    series = series.str.replace(r'\\times', ' x ', regex=True)


    #begin
    series = series.str.replace(
    r'\\begin\{array\}[^}]*\}(.*?)\\end\{array\}',
    r'\1',
    regex=True
    )

    series = series.str.replace(r'\\cdot', ' ', regex=True)
    series = series.str.replace(r'\\+', ' ', regex=True)
    series = series.str.replace(r'\n+', ' ', regex=True)
    series = series.str.replace(r'\s+', ' ', regex=True)
    series = series.str.replace(r'\\begin\{array\}', ' ', regex=True)
    series = series.str.replace(r'\\end\{array\}', ' ', regex=True)



    return series


def normalize_latex_whole(series, commands, max_iter=10):
    curr = series.copy()

    for _ in range(max_iter):
        prev = curr

        for cmd in commands:
          curr = normalize_latex(curr, cmd)

        if curr.equals(prev):
            break

    curr = normalize_latex_operators(curr)
    return curr


cmd = ['text', 'mathbf', 'boldsymbol', 'mathrm', 'operatorname','overline','bar']
df['questions'] = normalize_latex_whole(df['questions'], cmd)
print(df['questions'].iloc[62542])



If the polarizing angle of a piece of glass
for green light is \( 54.74^{\circ}, \) then the angle of minimum deviation for an equilateral prism made of same glass is :
\( \left[\mathrm{GIVEN}, \tan 54.74^{\circ}=1.414\right] \)
A \( \cdot 45^{\circ} \)
B. 54.74
\( c \cdot 60 \)
\( D \cdot 30^{\circ} \)
If the polarizing angle of a piece of glass for green light is ( 54.74^{ circ}, ) then the angle of minimum deviation for an equilateral prism made of same glass is : ( [GIVEN, tan 54.74^{ circ}=1.414 ] ) A ( 45^{ circ} ) B. 54.74 ( c x 60 ) ( D x 30^{ circ} )


In [ ]:
df.sample(30)
# df.groupby("Subject").size()

,questions,Subject,matched_questions
40490,"What is the correct relationship between the ( p H^{ prime} s ) of isomolar solutions of sodium oxide ( (p H_{1} ), ) sodium sulphide ( (p H_{2} ), ) sodium selenide ( (p H_{3} . ) )and sodium telluride ( (p H_{4} ) ? ) A ( . p H_{1}<p H_{2}<p H_{3}<p H_{4} ) в. ( p H_{1}>p H_{2}>p H_{3}>p H_{4} ) c. ( p H_{1}<p H_{2}<p H_{3} approx p H_{4} ) D. ( p H_{1}>p H_{2} approx p H_{3}>p H_{4} )",Chemistry,False
40423,"On the ellipse, ( 9 x^{2}+25 y^{2}=225, ) find the point whose distance to the focus ( F_{1} ) is four times the distance to the other focus ( F_{2} ) ( A [-15, sqrt{63}] ) ( ^{В} ((-15)/(4), frac{ sqrt{63}}{2} ) ) ( ^{c} ((-15)/(4), frac{ sqrt{63}}{4} ) ) D. ( ((-15)/(2), frac{ sqrt{63}}{2} ) )",Maths,False
35542,"When we see an object, the image formed on the retina is (i) real (ii) virtual(iii) erect (iv) inverted A. (i) and (iv) only B. (i), (ii), (iii) c. (iv) only D. (ii) and (iv)",Physics,False
71328,"For the type of interactions : (I) covalent bond, (II) van der Waals' forces, and (III) hydrogen bonding, which represents the correct order of increasing stability? A. ( (I)<(I I I)<(I I) ) B. ( (I I)<(I I I)<(I) ) C. ( (I I I)<(I I)<(I) ) D. ( (I I)<(I)<(I I I) )",Chemistry,False
20435,"A container is filled with a radioactive substance for which the half-life is 2 days. A week later, when the container is opened, it contains ( 5 g ) of the substance. Approximately how may grams of the substances were initially placed in the container? A . 40 B. 60 c. 80 D. 100",Physics,False
10134,Each codon consists of how many nitrogen bases? A. four B. twenty c. three D. sixty four,Chemistry,False
122138,Which is the principal cation in the plasma of the blood A. Calcium B. Sodium c. Potassium D. Magnesium,Biology,False
16886,The fraction of ice that melts by mixing equal masses of ice at ( -10^{ circ} C ) and water at ( 60^{ circ} C ) is : A ( (6)/(11) ) в. ( (11)/(16) ) c. ( (5)/(16) ) D. ( (11)/(15) ),Physics,False
39668,Which impurity (as a salt) is associated with table salt obtained from sea-water? A. ( N a H C O_{3} ) B. ( M g C O_{3} ) c. ( M g C l_{2} ) D. NaI,Chemistry,False
75336,Assertion Sucrose is a reducing sugar and exhibits mutarotation. Reason Sucrose is a disaccharide consisting of glucose and fructose units. A. Statements A and R are true and ( R ) is the correct explanation of ( A ) B. Statements A and R are true and R is not the correct explanation of ( A ) C. Statements A is true and R is false. D. Statements A is false and R is true.,Chemistry,False


In [ ]:
#Flag questions with visual dependency





pattern = [
    r'\bfollowing\b',           # Word boundary for "following"
    r'\bshown\b',               # Exact word "shown"
    r'\bdiagram\b',
    r'\bfigure\b',
    r'fig\.\s*\d+',             # Fig. 1, Fig. 2, etc.
    r'figure\s*\d+',            # Figure 1, Figure 2, etc.
    r'circuit\s+shown',         # Circuit shown
    r'as\s+shown',              # As shown
    r'given\s+(below|above|in|figure|diagram)',  # Given below/above/in
    r'refer\s+to',              # Refer to
    r'shown\s+(in|below|above)',  # Shown in/below/above
]

combine_pattern = '|'.join(pattern)

df['matched_questions'] = df['questions'].str.contains(
    combine_pattern,
    case=False,
    regex=True,
    na=False
)


df["matched_questions"]



/tmp/ipykernel_723/1221657136.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['matched_questions'] = df['questions'].str.contains(


,matched_questions
0,False
1,True
2,False
3,False
4,True
...,...
122514,True
122515,False
122516,False
122517,False


In [ ]:
total = df.groupby('Subject').size()
matched = df['matched_questions'].groupby('Subject').size()

summary = pd.DataFrame({
    'questions': total,
    'matched_questions': matched
}).fillna(0)

summary['percentage'] = (summary['matched_questions'] / summary['questions']) * 100

print(summary)


KeyError: 'Subject'

In [ ]:
# Remove Biology questions
df_final = df[df["Subject"] != "Biology"]

# Remove figure/diagram-based questions
df_final = df_final[df_final["matched_questions"] == 0]

# Reset index
df_final = df_final.reset_index(drop=True)
df_final = df_final.drop(columns=["matched_questions"])


# Check results
print("Final number of questions:", len(df_final))
print(df_final["Subject"].value_counts())


Final number of questions: 89535
Subject
Physics      32112
Maths        28738
Chemistry    28685
Name: count, dtype: int64


In [ ]:
# Deduplicate by question text before embedding
df_final = df_final.drop_duplicates(subset=["questions"]).reset_index(drop=True)
print(df_final.duplicated(subset=["questions"]).sum())

print(f"After dedup: {len(df_final)} questions")

0
After dedup: 88824 questions


In [ ]:
df_final.sample(10)

,questions,Subject
41330,"A block of mass 2 kg slides on a rough surface at ( t=0, ) if speed is ( 2 m / s ). It stops after covering a distance of ( 20 cm ) because of friction. Find work done by the friction.",Physics
87750,"What is the energy band gap of silicon and germanium respectively in ( e V ) ? A . 1.1,0.7 в. 0.7,1.1 c. -0.7,-1.1 D. -1.1,-0.7",Physics
20406,"A manufacturer produces nuts and bolts. It takes 1 hour of work on machine ( A ) and 3 hours on machine ( B ) to produce a package of nuts. It takes 3 hours on machine ( A ) and 1 hour on machine ( B ) to produce a package of bolts. He earns a profit of ( R s .17 .50 ) per package on nuts and ( R s .7 .00 ) per package on bolts. How many packages of each should be produced each day so as to maximise his profit, if he operates his machines for at the most 12 hours a day?",Maths
34408,Determine of ( U ) is A . 13 B. 15 ( c .3 ) D. 2,Maths
59478,Universal gravitational constant ( G ) proportional to distance ( r ) between two masses as- A ( r^{2} ) B. ( c x 1 / r ) D. does not depend upon ( r ),Physics
61221,A body of mass ( 2 k g ) is lying on a floor The coefficient of static friction is 0.54 What will be the value of frictional force if the force is ( 2.8 N ) and ( g=10 m s^{-2} ? ) A . zero в. ( 2 N ) ( c .2 .8 N ) D. ( 8 N ),Physics
72927,"Assertion If ( alpha, beta ) are the roots of the equation ( 18 ( tan ^{-1} x )^{2}-9 pi tan ^{-1} x+ pi^{2}=0 ) ( then alpha+ beta= frac{4}{ sqrt{3}} ) Reason ( sec ^{2} ( cos ^{-1} ((1)/(4) ) )+ ) ( cosec^{2} ( sin ^{-1} ((1)/(5) ) )=41 ) A. Both Assertion and Reason are correct and Reason is the correct explanation for Assertion B. Both Assertion and Reason are correct but Reason is not the correct explanation for Assertion c. Assertion is correct but Reason is incorrect D. Assertion is incorrect but Reason is correct",Maths
5505,Which is more reactive nucleophile in polar protic solvent? ( A x F ) в. ( Cl ) ( c x B r ) D. I,Chemistry
34776,( O_{3}+K_{4} [F e (C N_{6} ) ] ) What is the product formed?,Chemistry
76290,The catalyst used for olefin polymerisation is: A. zieger-Natta Catalyst B. Raney Nickel Catalyst c. wilkinson Catalyst D. Merrifield Resin,Chemistry


In [ ]:
# Convert the current index into a column
df_final['question_id'] = df_final.index

# If you want to ensure they are strings (often better for IDs)
df_final['question_id'] = df_final.index.astype(str)
df_final

,questions,Subject,question_id
0,"If the area of two similar triangles are equal, then they are A . equilateral B. isosceles c. congruent D. not congruent",Maths,0
1,Electric current flows through: A. a conductor B. an insulator c. free space D. none of these,Physics,1
2,The sides of a right angled triangle are in A.P. The ratio of sides is A .1: 2: 3 B. 2:3:4 ( c x 3: 4: 5 ) ( D x 5: 8: 3 ),Maths,2
3,"If the mass of a body is ( M ) on the surface of the earth, the mass of the same body on the surface of the moon is A. ( M / 6 ) в. ( M ) ( c x 6 M ) D. zero",Physics,3
4,A particle of mass ( m ) is made to move with uniform speed ( v ) along the perimeter of a regular hexagon. The magnitude of impulse applied at each corner is: A . ( m v ) B. ( m v sqrt{3} ) c. ( (m v)/(2) ) D. ( frac{m v}{ sqrt{3}} ),Physics,4
...,...,...,...
88819,"The IUPAC name for given hydrocarbon is: A. neononane B. tetraethylcarbon c. 2 -ethylpentane D. 3,3 -diethylpentane",Chemistry,88819
88820,The energy of electron in the ground state of hydrogen atom is ( -13.6 eV ). The energy required for the transition from ( n=2 ) to ( n=3 ) will be A ( .2 e V ) в. ( 4 e V ) c. ( 1.89 e V ) D. ( 2.89 e V ),Physics,88820
88821,Light year is the A. light emitted by the sun in one year. B. the time taken by light to travel from sun to earth. C. the distance travelled by light in free space in one year. D. the time taken by earth to go once around the sun.,Physics,88821
88822,"In one average-life, A. half the active nuclei decay B. less than half the active nuclei decay. C. more than half the active nuclei decay. D. all the nuclei decay.",Physics,88822
